# deploy

> Update the running box to the pushed main: `reconcile-deploy`

In [ ]:
#| default_exp deploy

In [ ]:
#| hide
from nbdev.showdoc import *

The box pip-installs this app from GitHub main (baked into the boxrecipe
cloud-init), so **origin/main is the only deployable source** — the command
refuses to run when the local HEAD is ahead of it. After the reinstall the
recipe's two permission lines are re-applied verbatim: root's pip writes
files under whatever umask it inherits, and the service user reads the venv
via group only (the class of bug that bit discopipe and the first archive
push). In-place deploy and a fresh rebuild therefore converge on the same
state.

## `deploy_cmd`

In [ ]:
#| export
def deploy_cmd(
    host:str="doyu@box.ninjalabo.ai",  # ssh target (sudo-capable login user)
    deps:bool=False,                   # also reinstall dependencies
)->list:                               # ssh argv
    "Build the ssh argv: pip reinstall from GitHub main + recipe perms + restart."
    nd = "" if deps else " --no-deps"
    remote = (
        "set -e\n"
        f"sudo /opt/reconcile/bin/pip install --upgrade --force-reinstall{nd} git+https://github.com/doyu/reconcile-web.git\n"
        # the recipe's permission lines, verbatim: root's pip writes under its
        # umask, and the service user reads the venv via group only
        "sudo chown -R root:reconcile /opt/reconcile\n"
        "sudo chmod -R u=rwX,g=rX,o= /opt/reconcile\n"
        "sudo systemctl restart reconcile\n"
        "systemctl is-active reconcile\n")
    return ["ssh", host, remote]

In [ ]:
cmd = deploy_cmd()
assert cmd[0] == "ssh" and cmd[1] == "doyu@box.ninjalabo.ai" and len(cmd) == 3
remote = cmd[2]
# fail fast on any step
assert remote.startswith("set -e")
# reinstall from the only deployable source; deps stay pinned by default
assert "/opt/reconcile/bin/pip install --upgrade --force-reinstall --no-deps git+https://github.com/doyu/reconcile-web.git" in remote
assert "--no-deps" not in deploy_cmd(deps=True)[2]
# the recipe's permission lines, re-applied after root's pip wrote files
assert "chown -R root:reconcile /opt/reconcile" in remote
assert "chmod -R u=rwX,g=rX,o= /opt/reconcile" in remote
assert remote.index("pip install") < remote.index("chown") < remote.index("chmod")
# restart last, then report the unit state
assert remote.index("chmod") < remote.index("systemctl restart reconcile")
assert "systemctl is-active reconcile" in remote
# host override passes through
assert deploy_cmd(host="x@y")[1] == "x@y"

## `reconcile_deploy`

In [ ]:
#| export
import subprocess, sys
from pathlib import Path
from fastcore.script import call_parse

@call_parse
def reconcile_deploy(
    host:str="doyu@box.ninjalabo.ai",  # ssh target (sudo-capable login user)
    repo:str="~/reconcile-web",        # local checkout, used to check HEAD is pushed
    deps:bool=False,                   # also reinstall dependencies
):
    "Deploy the pushed main to the box: pip reinstall + recipe perms + restart"
    d = Path(repo).expanduser()
    if (d/".git").exists():
        run = lambda *a: subprocess.run(a, capture_output=True, text=True).stdout.split()
        local, remote = run("git", "-C", str(d), "rev-parse", "HEAD"), run("git", "-C", str(d), "ls-remote", "origin", "main")
        if local and remote and local[0] != remote[0]:
            sys.exit(f"local HEAD {local[0][:7]} != origin/main {remote[0][:7]} — push first (the box installs from GitHub main)")
    cmd = deploy_cmd(host, deps)
    print(f"deploying origin/main to {host}", file=sys.stderr)
    raise SystemExit(subprocess.run(cmd).returncode)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()